In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks, get_ranks_s
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *

In [3]:
import tsfresh
import json

In [4]:
SEED = 42
np.random.seed(SEED)

In [5]:
results = []
_, datasets, _ = get_global_const()

Parsing Bakeoff2017 models...

Parsing Bakeoff2021 models...

Parsing Bakeoff2023 models...

Parsing HIVE-COTEV2 models...



In [6]:
DATASETS_DIR = Path("data/datasets_metrics/datasets")
OUT_DIR = Path("data/datasets_features")

In [7]:
def load_dataset_json(dataset_name):
    
    json_path = DATASETS_DIR / f"{dataset_name}.json"
    
    with open(json_path, "r") as f:
        return json.load(f)

In [8]:
def extract_series_list(time_data):

    series_list = []
    
    for series in time_data["X"]:
        arr = np.asarray(series[0], dtype=float)
        series_list.append(arr)
        
    return series_list

In [9]:
def build_long_df_for_tsfresh(series_list, dataset_name):
    
    rows = []
    
    for sid, s in enumerate(series_list):
        
        for t, v in enumerate(s):
            rows.append({"dataset": dataset_name, "series_id": sid, "id": sid, "time": t, "v": float(v)})
            
    return pd.DataFrame(rows)

In [10]:
def compute_tsfresh_series_features(datasets, use_minimal=False):
    
    from tsfresh import extract_features
    from tsfresh.feature_extraction import MinimalFCParameters

    all_frames = []
    
    for ds in tqdm(datasets, disable=True):
        
        print(f"[TSFRESH] {ds}")
        
        time_data = load_dataset_json(ds)
        
        series_list = extract_series_list(time_data)
        
        df_long = build_long_df_for_tsfresh(series_list, ds)

        kwargs = dict(
            column_id="id",
            column_sort="time",
            column_value="v",
            disable_progressbar=True,
            n_jobs=0,
        )
        
        if use_minimal:
            kwargs["default_fc_parameters"] = MinimalFCParameters()

        df_feat = extract_features(df_long, **kwargs)

        df_feat = df_feat.reset_index().rename(columns={"index": "series_id"})
        df_feat.insert(0, "dataset", ds)

        all_frames.append(df_feat)

    return pd.concat(all_frames, ignore_index=True)

In [11]:
def compute_catch22_series_features(datasets):
    
    try:
        from pycatch22 import catch22_all
    except ImportError:
        raise ImportError(
            "Install Catch22 first: pip install pycatch22\n"
            "Then import should work as: from catch22 import catch22_all"
        )

    rows = []
    
    for ds in tqdm(datasets, disable=True):
        
        print(f"[Catch22] {ds}")
        
        time_data = load_dataset_json(ds)
        
        series_list = extract_series_list(time_data)

        for sid, s in enumerate(series_list):
            out = catch22_all(s)
            feat = dict(zip(out["names"], out["values"]))
            feat["dataset"] = ds
            feat["series_id"] = sid
            rows.append(feat)

    return pd.DataFrame(rows)

In [12]:
def spectral_entropy(x, eps=1e-12):

    x = np.asarray(x, dtype=float)
    
    if x.size < 2:
        return 0.0
    
    x = x - x.mean()

    p = np.abs(np.fft.rfft(x)) ** 2
    p_sum = p.sum()
    
    if p_sum <= eps:
        return 0.0
    
    p = p / p_sum
    
    ent = -(p * np.log(p + eps)).sum()
    
    return float(ent)


def autocorr_lag(x, lag):
    
    x = np.asarray(x, dtype=float)
    
    if x.size <= lag or x.size < 2:
        return 0.0
    
    x0 = x[:-lag] - x[:-lag].mean()
    
    x1 = x[lag:] - x[lag:].mean()
    
    denom = (np.sqrt((x0**2).sum()) * np.sqrt((x1**2).sum()))
    
    if denom == 0:
        return 0.0
    
    return float((x0 * x1).sum() / denom)


def compute_summary_series_features(datasets, max_lag=5):
    
    from scipy.stats import skew, kurtosis

    rows = []
    
    for ds in tqdm(datasets, disable=True):
        
        print(f"[Summary] {ds}")
        
        time_data = load_dataset_json(ds)
        
        series_list = extract_series_list(time_data)

        for sid, s in enumerate(series_list):
            
            s = np.asarray(s, dtype=float)
            
            feat = {
                "dataset": ds,
                "series_id": sid,
                "len": int(s.size),
                "mean": float(np.mean(s)) if s.size else 0.0,
                "std": float(np.std(s)) if s.size else 0.0,
                "min": float(np.min(s)) if s.size else 0.0,
                "max": float(np.max(s)) if s.size else 0.0,
                "median": float(np.median(s)) if s.size else 0.0,
                "q25": float(np.quantile(s, 0.25)) if s.size else 0.0,
                "q75": float(np.quantile(s, 0.75)) if s.size else 0.0,
                "iqr": float(np.quantile(s, 0.75) - np.quantile(s, 0.25)) if s.size else 0.0,
                "skew": float(skew(s)) if s.size >= 3 else 0.0,
                "kurtosis": float(kurtosis(s)) if s.size >= 4 else 0.0,
                "spectral_entropy": spectral_entropy(s),
            }
            
            for lag in range(1, max_lag + 1):
                feat[f"acf_{lag}"] = autocorr_lag(s, lag)
                
            rows.append(feat)

    return pd.DataFrame(rows)


In [13]:
def pad_or_trim(x, target_len):
    
    x = np.asarray(x, dtype=float)
    
    if x.size == target_len:
        return x
    
    if x.size > target_len:
        return x[:target_len]
    

    pad_val = x[-1] if x.size else 0.0
    
    return np.pad(x, (0, target_len - x.size), mode="constant", constant_values=pad_val)

In [ ]:
def compute_minirocket_series_features(
    datasets,
    target_len=512,
    fit_sample_per_dataset=50,
    batch_size=512,
):

    import random
    import numpy as np
    import pandas as pd

    from sktime.transformations.panel.rocket import MiniRocketMultivariate

    def pad_or_trim(x, L):
        
        x = np.asarray(x, dtype=np.float32)
        
        if x.size == L:
            return x
        
        if x.size > L:
            return x[:L]
        
        pad_val = x[-1] if x.size else np.float32(0.0)
        
        return np.pad(x, (0, L - x.size), mode="constant", constant_values=pad_val).astype(np.float32)

    fit_series = []
    
    for ds in datasets:
        
        time_data = load_dataset_json(ds)
        series_list = extract_series_list(time_data)
        idx = list(range(len(series_list)))
        random.shuffle(idx)
        
        idx = idx[: min(fit_sample_per_dataset, len(idx))]
        
        for i in idx:
            fit_series.append(pad_or_trim(series_list[i], target_len))

    if not fit_series:
        raise ValueError("No series found to fit MiniRocket.")

    X_fit = np.stack(fit_series, axis=0)
    X_fit = np.nan_to_num(X_fit, nan=0.0, posinf=0.0, neginf=0.0)
    X_fit = np.ascontiguousarray(X_fit, dtype=np.float32)
    X_fit = X_fit[:, None, :]

    print(f"[MiniRocket] fitting on {X_fit.shape[0]} sampled series, len={target_len}")
    transformer = MiniRocketMultivariate(random_state=SEED)
    transformer.fit(X_fit)

    all_frames = []
    
    for ds in datasets:
        
        print(f"[MiniRocket] transform {ds}")
        
        time_data = load_dataset_json(ds)
        
        series_list = extract_series_list(time_data)
        

        X = np.stack([pad_or_trim(s, target_len) for s in series_list], axis=0)  # (n, L)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        X = np.ascontiguousarray(X, dtype=np.float32)
        X = X[:, None, :]

        feats_batches = []
        
        for start in range(0, X.shape[0], batch_size):
            xb = X[start:start + batch_size]
            fb = transformer.transform(xb)
            feats_batches.append(fb)

        X_feat = pd.concat(feats_batches, ignore_index=True)
        X_feat.insert(0, "series_id", np.arange(X_feat.shape[0], dtype=int))
        X_feat.insert(0, "dataset", ds)
        all_frames.append(X_feat)

    return pd.concat(all_frames, ignore_index=True)


In [16]:
# df_tsf = compute_tsfresh_series_features(datasets, use_minimal=False)
# out1 = OUT_DIR / "tsfresh_series_features.csv"
# df_tsf.to_csv(out1, index=False)
# print(f"Saved: {out1}  shape={df_tsf.shape}")

In [17]:
# df_c22 = compute_catch22_series_features(datasets)
# out2 = OUT_DIR / "catch22_series_features.csv"
# df_c22.to_csv(out2, index=False)
# print(f"Saved: {out2}  shape={df_c22.shape}")

In [18]:
# df_sum = compute_summary_series_features(datasets, max_lag=5)
# out3 = OUT_DIR / "summary_series_features.csv"
# df_sum.to_csv(out3, index=False)
# print(f"Saved: {out3}  shape={df_sum.shape}")

In [ ]:
# df_mr = compute_minirocket_series_features(datasets, target_len=512, fit_sample_per_dataset=50)
# out4 = OUT_DIR / "minirocket_series_features.csv"
# df_mr.to_csv(out4, index=False)
# print(f"Saved: {out4}  shape={df_mr.shape}")

[MiniRocket] fitting on 5580 sampled series, len=512
[MiniRocket] transform Adiac
[MiniRocket] transform ArrowHead
[MiniRocket] transform Beef
[MiniRocket] transform BeetleFly
[MiniRocket] transform BirdChicken
[MiniRocket] transform Car
[MiniRocket] transform CBF
[MiniRocket] transform ChlorineConcentration
[MiniRocket] transform CinCECGTorso
[MiniRocket] transform Coffee
[MiniRocket] transform Computers
[MiniRocket] transform CricketX
[MiniRocket] transform CricketY
[MiniRocket] transform CricketZ
[MiniRocket] transform DiatomSizeReduction
[MiniRocket] transform DistalPhalanxOutlineAgeGroup
[MiniRocket] transform DistalPhalanxOutlineCorrect
[MiniRocket] transform DistalPhalanxTW
[MiniRocket] transform Earthquakes
[MiniRocket] transform ECG200
[MiniRocket] transform ECG5000
[MiniRocket] transform ECGFiveDays
[MiniRocket] transform ElectricDevices
[MiniRocket] transform FaceAll
[MiniRocket] transform FaceFour
[MiniRocket] transform FacesUCR
[MiniRocket] transform FiftyWords
[MiniRocket